### Analysis of MWPM decoding graph

In [98]:
import numpy as np
import stim
from surface_code_stim import SurfaceCode
import pymatching
import plotly.graph_objects as go

1. Build a circuit-level noise circuit for the surface code using Stim

In [99]:
# Build the circuit manually, using custom error channels, to have more control over the error model.
hardware_params = {
        'T1': 250e-6,  # in seconds
        'T2': 250e-6,  # in seconds
        'duration_1q_gate': 20e-9,  # in seconds
        'duration_2q_gate': 30e-9,  # in seconds
        'state_prep_error': 1e-1,
        'measurement_error': 1e-1,
        'gate_error_1q': 1e-1,
        'gate_error_2q': 1e-1,
        'duration_readout': 500e-9
    }

sc = SurfaceCode(
        hardware_params=hardware_params,
        distance=3,
        n_rounds=3,
        ad=False,
        crosstalk=False,
        default_2q_gate='CX',
        fidelity_incl_decoherence=False,
        memory_type='z',
        multiplier=1.0,
        idle_depol=True
    )

sc_circuit_custom = sc.build_surface_code_circuit()

# Build the circuit using stim's built-in generator, which has a fixed error model.
sc_circuit_builtin = stim.Circuit.generated("surface_code:rotated_memory_z",
                                distance=3,
                                rounds=3,
                                after_clifford_depolarization=0.1,
                                before_measure_flip_probability=0.1,
                                after_reset_flip_probability=0.1,
                                 before_round_data_depolarization=0.1)

2. Build the detector error model (DEM), and analyze the difference between the two options of circuits.

In [100]:
dem_custom = sc_circuit_custom.detector_error_model(decompose_errors=True)
dem_builtin = sc_circuit_builtin.detector_error_model(decompose_errors=True)

print("Custom circuit error model:")
print(dem_custom)
print(f"Number of errors: {len(dem_custom)}")
print(f"Number of detectors: {dem_custom.num_detectors}")
print(f"Number of observables: {dem_custom.num_observables}")

print("\nBuilt-in circuit error model:")
print(dem_builtin)
print(f"Number of errors: {len(dem_builtin)}")
print(f"Number of detectors: {dem_builtin.num_detectors}")
print(f"Number of observables: {dem_builtin.num_observables}")

Custom circuit error model:
error(0.2160309899247322) D0 D2
error(0.3293661770569947) D0 D4
error(0.1723434499131525) D0 L0
error(0.1723434499131525) D1 D2
error(0.2233445925925927) D1 D3
error(0.02741843737473918) D1 D4
error(0.2852823703703704) D1 D5
error(0.1009777777777778) D1 D6
error(0.3428768756530972) D1 L0
error(0.3712939478440677) D2
error(0.05333333333333336) D2 D4
error(0.2852823703703704) D2 D6
error(0.2673892345679013) D3
error(0.05333333333333336) D3 D5
error(0.02741843737473918) D3 D6
error(0.3293661770569947) D3 D7
error(0.2872291024469022) D4 D6
error(0.3293661770569947) D4 D8
error(0.2935804725231144) D4 L0
error(0.02741843737473918) D4 L0 ^ D0 L0
error(0.02741843737473918) D4 L0 ^ D1 L0
error(0.2092365432098766) D5 D6
error(0.2872291024469024) D5 D7
error(0.02741843737473918) D5 D8
error(0.2852823703703704) D5 D9
error(0.1009777777777778) D5 D10
error(0.4258325564394715) D5 L0
error(0.05333333333333336) D5 L0 ^ D1 L0
error(0.4258325564394714) D6
error(0.053333333333

In [101]:
def extract_syndrome_from_circuit(circuit, n_shots=1, seed=None):
    if seed is not None:
        sampler = circuit.compile_detector_sampler(seed=seed)
    else:
        sampler = circuit.compile_detector_sampler()

    syndrome, true_obs = sampler.sample(shots=n_shots, separate_observables=True)

    return syndrome, true_obs

In [102]:
def plot_mwpm_solution_3d(circuit, matching, syndrome, true_obs, pred_obs, solution_edges, show_boundary=True):
    '''
    Plots the matching graph in 3D, with edges colored by whether they are part of the MWPM solution or not.

    Args:
     - circuit: the stim.Circuit object, used to get detector coordinates
     - matching: the pymatching.Matching object, which contains the graph structure
     - syndrome: the observed syndrome (1D array of 0/1)
     - true_obs: the true observable values (1D array of 0/1)
     - solution_edges: the edges selected by MWPM (array of shape (k,2) with pairs of node indices)
     - show_boundary: whether to include edges connected to boundary nodes

    Returns:
     - A dictionary containing the matching graph, syndrome, solution edges, and the Plotly figure
    '''

    # Export to networkx-like structure for visualization
    G = matching.to_networkx()
    fired = set(np.where(syndrome == 1)[0].tolist())

    # MWPM solution edges
    sol_set = {tuple(sorted(map(int, e))) for e in solution_edges}

    # Coordinates
    det_xyz = circuit.get_detector_coordinates()

    # boundary node(s): PyMatching uses explicit boundary nodes; in many setups you’ll see -1
    # We handle any nodes not in det_xyz by placing them off to the side.
    all_nodes = list(G.nodes())
    xs = [det_xyz[n][0] for n in det_xyz if n in all_nodes]
    ys = [det_xyz[n][1] for n in det_xyz if n in all_nodes]
    zs = [det_xyz[n][2] for n in det_xyz if n in all_nodes]
    cx = float(np.mean(xs)) 
    cy = float(np.mean(ys)) 
    cz = float(np.mean(zs)) 
    minx = float(np.min(xs)) if xs else -1.0

    pos = {}
    for n in all_nodes:
        if n in det_xyz:
            pos[n] = det_xyz[n]
        else:
            # boundary / auxiliary nodes
            pos[n] = (minx - 2.0, cy, cz)

    # Build Plotly traces
    all_x, all_y, all_z = [], [], []
    sel_x, sel_y, sel_z = [], [], []
    mid_x, mid_y, mid_z, mid_txt = [], [], [], []

    for u, v, data in G.edges(data=True):
        u_i, v_i = int(u), int(v)

        # Optionally hide boundary edges
        if not show_boundary:
            if (u not in det_xyz) or (v not in det_xyz):
                continue

        x0, y0, z0 = pos[u]
        x1, y1, z1 = pos[v]

        w = data.get("weight", None)
        p = data.get("error_probability", None)

        mid_x.append((x0 + x1) / 2)
        mid_y.append((y0 + y1) / 2)
        mid_z.append((z0 + z1) / 2)
        mid_txt.append(f"{u_i}—{v_i}<br>weight={w}<br>p={p}")

        e = tuple(sorted((u_i, v_i)))
        if e in sol_set:
            sel_x += [x0, x1, None]
            sel_y += [y0, y1, None]
            sel_z += [z0, z1, None]
        else:
            all_x += [x0, x1, None]
            all_y += [y0, y1, None]
            all_z += [z0, z1, None]

    node_x, node_y, node_z, node_txt = [], [], [], []
    fired_x, fired_y, fired_z, fired_txt = [], [], [], []

    for n in all_nodes:
        x, y, z = pos[n]
        label = f"D{n}" if n in det_xyz else f"boundary/{n}"
        if n in fired:
            fired_x.append(x); fired_y.append(y); fired_z.append(z)
            fired_txt.append(label + " (fired)")
        else:
            node_x.append(x); node_y.append(y); node_z.append(z)
            node_txt.append(label)

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode="lines",
        line=dict(width=2),
        name="graph edges",
        hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter3d(
        x=sel_x, y=sel_y, z=sel_z,
        mode="lines",
        line=dict(width=7),
        name="MWPM selected edges",
        hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter3d(
        x=mid_x, y=mid_y, z=mid_z,
        mode="markers",
        marker=dict(size=2, opacity=0.0),
        text=mid_txt,
        hoverinfo="text",
        name="edge info (hover)",
    ))
    fig.add_trace(go.Scatter3d(
        x=node_x, y=node_y, z=node_z,
        mode="markers",
        marker=dict(size=4),
        text=node_txt,
        hoverinfo="text",
        name="nodes",
    ))
    fig.add_trace(go.Scatter3d(
        x=fired_x, y=fired_y, z=fired_z,
        mode="markers",
        marker=dict(size=7),
        text=fired_txt,
        hoverinfo="text",
        name="fired detectors",
    ))

    fig.update_layout(
        title=f"MWPM on matching graph (true obs={1 if true_obs else 0}, pred obs={pred_obs[0][0]})",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="t (round)",
            aspectmode="data",
        ),
        legend=dict(itemsizing="constant"),
    )
    fig.show()

    return G, fig


In [111]:
enable_correlations = True

syndrome, true_obs = extract_syndrome_from_circuit(sc_circuit_builtin, n_shots=1)
matching = pymatching.Matching.from_detector_error_model(dem_builtin, enable_correlations=enable_correlations)
pred_obs = matching.decode_batch(syndrome)
solution_edges = matching.decode_to_edges_array(syndrome)

G, fig = plot_mwpm_solution_3d(sc_circuit_builtin, matching, syndrome, true_obs, pred_obs, solution_edges, show_boundary=True)
for u, v in solution_edges:
    data = G.get_edge_data(int(u), int(v))
    print(u, v, data)


6 -1 None
10 -1 None
11 -1 None
15 23 {'fault_ids': set(), 'weight': 1.4016917094987276, 'error_probability': 0.1975477999198331}
16 20 {'fault_ids': set(), 'weight': 1.4016917094987278, 'error_probability': 0.1975477999198331}
17 -1 None
18 22 {'fault_ids': set(), 'weight': 1.3001882687909334, 'error_probability': 0.21413333333333331}
19 -1 None
21 -1 None
